In [1]:
# Cargar dependencias y preparar rutas

import json
from pathlib import Path
import numpy as np
from sentence_transformers import SentenceTransformer

DATA_PATH = Path("data/chunks_legal/chunks_final_all_sources.jsonl")
OUT_DIR = Path("processed/vectorstore")
OUT_DIR.mkdir(parents=True, exist_ok=True)

EMB_PATH = OUT_DIR / "embeddings.npy"
META_PATH = OUT_DIR / "chunks_meta.jsonl"

print("Rutas preparadas correctamente")

/Users/danielocando/IA/KeepCoding/BOOTCAMP IA/NormaBot TFM BOOTCAMP IA IV/proyecto-final/.venv_rag/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Rutas preparadas correctamente


In [ ]:
# Cargar modelo de embeddings

MODEL_NAME = "intfloat/multilingual-e5-base"
model = SentenceTransformer(MODEL_NAME)

print("Modelo OK:", MODEL_NAME)

In [4]:
# Cargar dependencias y preparar rutas

from pathlib import Path
from sentence_transformers import SentenceTransformer

# Estamos dentro de notebooks/, así que subimos un nivel
DATA_PATH = Path("../data/chunks_legal/chunks_final_all_sources.jsonl")

OUT_DIR = Path("../processed/vectorstore")
OUT_DIR.mkdir(parents=True, exist_ok=True)

EMB_PATH = OUT_DIR / "embeddings.npy"
META_PATH = OUT_DIR / "chunks_meta.jsonl"

print("Rutas preparadas correctamente")
print("Dataset existe:", DATA_PATH.exists())

Rutas preparadas correctamente
Dataset existe: True


In [5]:
# Cargar dataset (jsonl) en memoria

texts = []
metadata = []

with DATA_PATH.open("r", encoding="utf-8") as f:
    for line in f:
        record = json.loads(line)
        texts.append(record["text"])
        metadata.append(record)

print("Chunks cargados:", len(texts))
print("Ejemplo de texto:")
print(texts[0][:300])

Chunks cargados: 699
Ejemplo de texto:
Disposición adicional primera
Disposición adicional segunda
Disposición adicional tercera
Disposición adicional cuarta
Disposición adicional quinta
Disposición derogatoria única
Disposición final primera
Disposición final segunda
Disposición final tercera
Disposición final cuarta
Disposición final q


In [ ]:
# Generar embeddings de todos los chunks (prefijo 'passage: ' requerido por e5)

print("Generando embeddings...")

prefixed = [f"passage: {t}" for t in texts]
embeddings = model.encode(
    prefixed,
    batch_size=32,
    show_progress_bar=True,
    convert_to_numpy=True
)

print("Embeddings generados")
print("Shape:", embeddings.shape)

In [7]:
# Guardar embeddings y metadatos en disco

np.save(EMB_PATH, embeddings)

with META_PATH.open("w", encoding="utf-8") as f:
    for record in metadata:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")

print("Vectorstore guardado correctamente")
print("Embeddings:", EMB_PATH)
print("Metadata:", META_PATH)

Vectorstore guardado correctamente
Embeddings: ../processed/vectorstore/embeddings.npy
Metadata: ../processed/vectorstore/chunks_meta.jsonl


In [ ]:
# Cargar embeddings desde disco y probar búsqueda semántica básica

import numpy as np

# Cargar embeddings guardados
embeddings = np.load(EMB_PATH)

def search(query, top_k=5):
    # Prefijo 'query: ' requerido por e5
    query_embedding = model.encode(f"query: {query}", convert_to_numpy=True)
    
    # Normalizar para cosine similarity
    emb_norm = embeddings / np.linalg.norm(embeddings, axis=1, keepdims=True)
    query_norm = query_embedding / np.linalg.norm(query_embedding)
    
    scores = emb_norm @ query_norm
    top_indices = np.argsort(scores)[-top_k:][::-1]
    
    return [(scores[i], texts[i]) for i in top_indices]

# Prueba
results = search("¿Qué es un sistema de alto riesgo según el Reglamento de IA?")

for score, text in results:
    print("Score:", round(float(score), 4))
    print(text[:400])
    print("-" * 60)

In [ ]:
# Función de búsqueda semántica (retrieve)

import numpy as np

def search(query: str, top_k: int = 5):
    # Prefijo 'query: ' requerido por e5
    query_emb = model.encode(f"query: {query}", convert_to_numpy=True)

    # Normalizamos para coseno = producto punto
    E = embeddings / np.linalg.norm(embeddings, axis=1, keepdims=True)
    q = query_emb / np.linalg.norm(query_emb)

    scores = E @ q
    idx = np.argsort(scores)[::-1][:top_k]

    results = []
    for i in idx:
        rec = metadata[i]
        results.append(
            {
                "score": float(scores[i]),
                "text": texts[i],
                "source": rec.get("source"),
                "doc_title": rec.get("doc_title"),
                "unit_type": rec.get("unit_type"),
                "unit_title": rec.get("unit_title"),
            }
        )
    return results

# Prueba rápida
hits = search("sistemas de alto riesgo", top_k=5)
for h in hits:
    print(f"Score: {h['score']:.4f}")
    print(h.get("unit_title") or h.get("doc_title") or "")
    print(h["text"][:250])
    print("-" * 60)

In [ ]:
# Resumen técnico (embeddings + vectorización + retrieve)

print("=== Resumen técnico ===")
print(f"Modelo de embeddings: {MODEL_NAME}")
print(f"Nº chunks: {len(texts)}")
print(f"Dimensión embedding: {embeddings.shape[1]}")
print(f"Embeddings (matriz): {embeddings.shape}  -> (n_chunks, dim)")

print("\nVectorstore en disco:")
print(f"- ChromaDB: {CHROMA_DIR}")

print("\nBúsqueda semántica (retrieve):")
print("- Similaridad: coseno (cosine similarity)")
print("- Prefijos e5: 'passage: ' para documentos, 'query: ' para consultas")
print("- Flujo: query -> embedding('query: '+query) -> cos_sim vs embeddings -> top_k resultados")

print("\nNota:")
print("- Los 'scores' mostrados son similitudes coseno (cuanto más alto, más parecido semánticamente).")

In [11]:
# Verificar si AESIA está dentro del corpus actual (chunks_final_all_sources.jsonl)

# 1) Listar fuentes únicas en metadata
sources = sorted({m.get("source") for m in metadata if isinstance(m, dict)})

print("Fuentes detectadas:", sources)

# 2) Buscar si aparece AESIA (por nombre exacto o parecido)
candidatas = [s for s in sources if s and "aesia" in s.lower()]
print("Fuentes que contienen 'aesia':", candidatas)

# 3) Contar cuántos chunks vienen de AESIA (si existe)
if candidatas:
    aesia_source = candidatas[0]
    n_aesia = sum(1 for m in metadata if isinstance(m, dict) and m.get("source") == aesia_source)
    print("Chunks AESIA:", n_aesia)
else:
    print("No se detectó AESIA en 'source'. Puede estar en otro campo (file/doc_title).")

# 4) Fallback: buscar 'aesia' en otros campos típicos
fields = ["file", "doc_title", "unit_title"]
hits = {f: 0 for f in fields}

for m in metadata:
    if not isinstance(m, dict):
        continue
    for f in fields:
        v = m.get(f)
        if isinstance(v, str) and "aesia" in v.lower():
            hits[f] += 1

print("Búsqueda 'aesia' por campos:", hits)

Fuentes detectadas: ['aesia', 'boe', 'eu_ai_act', 'lopd_rgpd']
Fuentes que contienen 'aesia': ['aesia']
Chunks AESIA: 75
Búsqueda 'aesia' por campos: {'file': 0, 'doc_title': 0, 'unit_title': 0}


In [12]:
# Crear cliente persistente de Chroma (vectorstore local)
from pathlib import Path
import chromadb

CHROMA_DIR = Path("vectorstore/chroma")
CHROMA_DIR.mkdir(parents=True, exist_ok=True)

client = chromadb.PersistentClient(path=str(CHROMA_DIR))
collection = client.get_or_create_collection(name="normabot_legal_chunks")

print("Chroma OK")
print("Path:", CHROMA_DIR.resolve())
print("Collection:", collection.name)
print("Count:", collection.count())

Chroma OK
Path: /Users/danielocando/IA/KeepCoding/BOOTCAMP IA/NormaBot TFM BOOTCAMP IA IV/proyecto-final/notebooks/vectorstore/chroma
Collection: normabot_legal_chunks
Count: 0


In [13]:
# Rutas definitivas (NO guardar nada en notebooks/ excepto .ipynb)
from pathlib import Path

# Dataset (se queda en data/)
DATA_PATH = Path("../data/chunks_legal/chunks_final_all_sources.jsonl")

# Todo lo generado va a processed/
VSTORE_DIR = Path("../processed/vectorstore")
VSTORE_DIR.mkdir(parents=True, exist_ok=True)

EMB_PATH = VSTORE_DIR / "embeddings.npy"
META_PATH = VSTORE_DIR / "chunks_meta.jsonl"
CHROMA_DIR = VSTORE_DIR / "chroma"

print("DATA_PATH:", DATA_PATH, "exists:", DATA_PATH.exists())
print("VSTORE_DIR:", VSTORE_DIR)
print("EMB_PATH:", EMB_PATH, "exists:", EMB_PATH.exists())
print("META_PATH:", META_PATH, "exists:", META_PATH.exists())
print("CHROMA_DIR:", CHROMA_DIR, "exists:", CHROMA_DIR.exists())

DATA_PATH: ../data/chunks_legal/chunks_final_all_sources.jsonl exists: True
VSTORE_DIR: ../processed/vectorstore
EMB_PATH: ../processed/vectorstore/embeddings.npy exists: True
META_PATH: ../processed/vectorstore/chunks_meta.jsonl exists: True
CHROMA_DIR: ../processed/vectorstore/chroma exists: True


In [15]:
# Comprobar colección y count en Chroma (desde el notebook)

import chromadb
from pathlib import Path

CHROMA_DIR = Path("../processed/vectorstore/chroma")
COLLECTION_NAME = "normabot_legal_chunks"

print("Chroma DIR:", CHROMA_DIR.resolve())
print("Exists:", CHROMA_DIR.exists())

client = chromadb.PersistentClient(path=str(CHROMA_DIR))

collections = client.list_collections()
print("Collections:", [c.name for c in collections])

collection = client.get_or_create_collection(name=COLLECTION_NAME)

print("Collection:", collection.name)
print("Count:", collection.count())

Chroma DIR: /Users/danielocando/IA/KeepCoding/BOOTCAMP IA/NormaBot TFM BOOTCAMP IA IV/proyecto-final/processed/vectorstore/chroma
Exists: True
Collections: ['normabot_legal_chunks']
Collection: normabot_legal_chunks
Count: 0


In [16]:
# Cargar embeddings y metadata (desde processed/vectorstore)

import json
from pathlib import Path

import numpy as np
import chromadb

VSTORE_DIR = Path("../processed/vectorstore")
EMB_PATH = VSTORE_DIR / "embeddings.npy"
META_PATH = VSTORE_DIR / "chunks_meta.jsonl"
CHROMA_DIR = VSTORE_DIR / "chroma"

COLLECTION_NAME = "normabot_legal_chunks"

print("EMB_PATH:", EMB_PATH, "exists:", EMB_PATH.exists())
print("META_PATH:", META_PATH, "exists:", META_PATH.exists())
print("CHROMA_DIR:", CHROMA_DIR, "exists:", CHROMA_DIR.exists())

embeddings = np.load(EMB_PATH)
metadata = []
texts = []

with META_PATH.open("r", encoding="utf-8") as f:
    for line in f:
        rec = json.loads(line)
        metadata.append(rec)
        texts.append(rec.get("text", ""))

print("Embeddings shape:", embeddings.shape)
print("Metadata:", len(metadata))
print("Texts:", len(texts))
print("Ejemplo texto:", texts[0][:200] if texts else "N/A")

EMB_PATH: ../processed/vectorstore/embeddings.npy exists: True
META_PATH: ../processed/vectorstore/chunks_meta.jsonl exists: True
CHROMA_DIR: ../processed/vectorstore/chroma exists: True
Embeddings shape: (699, 384)
Metadata: 699
Texts: 699
Ejemplo texto: Disposición adicional primera
Disposición adicional segunda
Disposición adicional tercera
Disposición adicional cuarta
Disposición adicional quinta
Disposición derogatoria única
Disposición final prim


In [17]:
# Poblar ChromaDB (upsert) con ids + documents + embeddings + metadatos

client = chromadb.PersistentClient(path=str(CHROMA_DIR))
col = client.get_or_create_collection(name=COLLECTION_NAME)

print("Antes - Count:", col.count())

# ids estables: si ya tienes 'chunk_id' úsalo; si no, generamos ids por índice
ids = []
metas = []

for i, rec in enumerate(metadata):
    cid = rec.get("chunk_id") or rec.get("id") or f"chunk_{i}"
    ids.append(str(cid))

    # guardamos metadata SIN el campo text para no duplicar (doc va en documents)
    m = dict(rec)
    m.pop("text", None)
    metas.append(m)

# upsert por batches para evitar problemas de memoria
batch_size = 200
n = len(ids)

for start in range(0, n, batch_size):
    end = min(start + batch_size, n)
    col.upsert(
        ids=ids[start:end],
        documents=texts[start:end],
        embeddings=embeddings[start:end].tolist(),
        metadatas=metas[start:end],
    )
    print(f"Upsert {start}:{end} OK")

print("Después - Count:", col.count())

Antes - Count: 0
Upsert 0:200 OK
Upsert 200:400 OK
Upsert 400:600 OK
Upsert 600:699 OK
Después - Count: 699


In [18]:
# === Punto 5 (parte 1): Refino retrieval ajustando k y comparando resultados ===
# Objetivo: ver si con k=3/5/8/12 los primeros resultados son más útiles para las queries.

import numpy as np

def chroma_search(query: str, k: int = 5, where: dict | None = None):
    res = col.query(
        query_texts=[query],
        n_results=k,
        where=where,  # ej: {"source": "aesia"}
        include=["documents", "metadatas", "distances"]
    )
    docs = res["documents"][0]
    metas = res["metadatas"][0]
    dists = res["distances"][0]  # en Chroma suele ser "distance" (menor = más parecido)
    return list(zip(dists, docs, metas))

def pretty_print(results, max_chars=350):
    for i, (dist, doc, meta) in enumerate(results, 1):
        src = meta.get("source", meta.get("fuente", "N/A"))
        title = meta.get("doc_title", meta.get("title", meta.get("unit_title", "N/A")))
        art = meta.get("article", meta.get("articulo", ""))
        print(f"\n#{i}  distance={dist:.4f}  source={src}  title={title}  {('art='+str(art)) if art else ''}")
        print(doc[:max_chars].replace("\n", " ").strip())
        print("-" * 80)

# Queries de prueba
test_queries = [
    "¿Qué requisitos tiene un sistema de IA de alto riesgo según el Reglamento de IA?",
    "¿Qué dice sobre sistema de gestión de riesgos y documentación?",
    "¿Qué obligaciones de información tiene el proveedor?",
]

for q in test_queries:
    print("\n" + "="*100)
    print("QUERY:", q)
    for k in [3, 5, 8, 12]:
        print("\n" + "-"*60)
        print(f"Top-k = {k}")
        results = chroma_search(q, k=k)
        pretty_print(results, max_chars=260)


QUERY: ¿Qué requisitos tiene un sistema de IA de alto riesgo según el Reglamento de IA?

------------------------------------------------------------
Top-k = 3


/Users/danielocando/.cache/chroma/onnx_models/all-MiniLM-L6-v2/onnx.tar.gz: 100%|██████████| 79.3M/79.3M [00:12<00:00, 6.55MiB/s]



#1  distance=2.8709  source=aesia  title=MINISTERIO PARA LA  
MINISTERIO PARA LA TRANSFORMACIÓN DIGITAL Y DE LA FUNCIÓN PÚBLICA SECRETARÍA DE ESTADO DE DIGITALIZACIÓN E INTELIGENCIA ARTIFICIAL ESPAÑA DIGITAL 2030 E INTERNACIONALIZACIÓN Este documento es de carácter confidencial y tiene efectos meramente informativos. No
--------------------------------------------------------------------------------

#2  distance=3.3300  source=boe  title=BOE-A-2018-16673 Ley Orgánica 3/2018, de 5 de diciembre, de Protección de Datos Personales y garantía de los derechos digitales.  
Las actuaciones de investigación podrán realizarse a través de sistemas digitales que, mediante la videoconferencia u otro sistema similar, permitan la comunicación bidireccional y simultánea de imagen y sonido, la interacción visual, auditiva y verbal entre l
--------------------------------------------------------------------------------

#3  distance=3.6669  source=lopd_rgpd  title=RGPD_AEPD_consolidado_ES.pdf  
Artícu

In [19]:
# === Punto 5 (parte 2): Refino retrieval con filtros por metadata (por fuente) ===
# Objetivo: probar "solo AESIA", "solo EU AI Act", etc., para ver si mejora la precisión por tipo de pregunta.

def run_filtered_queries(query: str, k: int = 5):
    sources = ["eu_ai_act", "aesia", "boe", "lopd_rgpd"]
    print("\n" + "="*100)
    print("QUERY:", query)
    print("SIN FILTRO")
    pretty_print(chroma_search(query, k=k, where=None), max_chars=260)

    for s in sources:
        print("\n" + "-"*60)
        print(f"FILTRO source={s}")
        try:
            pretty_print(chroma_search(query, k=k, where={"source": s}), max_chars=260)
        except Exception as e:
            print(f"(No se pudo filtrar por source={s}. Error: {e})")

run_filtered_queries("¿Qué establece AESIA sobre evaluación o supervisión de sistemas de IA?", k=5)
run_filtered_queries("¿Qué obligaciones impone el Reglamento UE de IA sobre sistemas de alto riesgo?", k=5)
run_filtered_queries("¿Qué exige RGPD/LOPD sobre tratamiento de datos personales y bases legales?", k=5)


QUERY: ¿Qué establece AESIA sobre evaluación o supervisión de sistemas de IA?
SIN FILTRO

#1  distance=2.6608  source=aesia  title=MINISTERIO PARA LA  
MINISTERIO PARA LA TRANSFORMACIÓN DIGITAL Y DE LA FUNCIÓN PÚBLICA SECRETARÍA DE ESTADO DE DIGITALIZACIÓN E INTELIGENCIA ARTIFICIAL ESPAÑA DIGITAL 2030 E INTERNACIONALIZACIÓN Este documento es de carácter confidencial y tiene efectos meramente informativos. No
--------------------------------------------------------------------------------

#2  distance=2.9697  source=boe  title=BOE-A-2018-16673 Ley Orgánica 3/2018, de 5 de diciembre, de Protección de Datos Personales y garantía de los derechos digitales.  
Las actuaciones de investigación podrán realizarse a través de sistemas digitales que, mediante la videoconferencia u otro sistema similar, permitan la comunicación bidireccional y simultánea de imagen y sonido, la interacción visual, auditiva y verbal entre l
--------------------------------------------------------------------------

In [20]:
# === Retrieval con filtro por fuente ===
# Cambia el valor de "source" si quiero probar otra (aesia, eu_ai_act, boe, lopd_rgpd)

query = "¿Qué establece AESIA sobre evaluación o supervisión de sistemas de IA?"

results = chroma_search(
    query,
    k=5,
    where={"source": "aesia"}
)

pretty_print(results, max_chars=260)


#1  distance=2.6608  source=aesia  title=MINISTERIO PARA LA  
MINISTERIO PARA LA TRANSFORMACIÓN DIGITAL Y DE LA FUNCIÓN PÚBLICA SECRETARÍA DE ESTADO DE DIGITALIZACIÓN E INTELIGENCIA ARTIFICIAL ESPAÑA DIGITAL 2030 E INTERNACIONALIZACIÓN Este documento es de carácter confidencial y tiene efectos meramente informativos. No
--------------------------------------------------------------------------------

#2  distance=3.1961  source=aesia  title=0  
interpretación de los resultados de salida de los sistemas de IA por parte de los responsables del despliegue; En la guía de supervisión humana, se establecen que para que el responsable del despliegue pueda interpretar la información de salida se debe documen
--------------------------------------------------------------------------------

#3  distance=3.3874  source=aesia  title=0  
Una descripción detallada del sistema establecido para evaluar el funcionamiento del sistema de IA en la fase posterior a la comercialización , de conformidad con

In [21]:
# === Evaluación formal de k ===
# Objetivo: medir si top-3 / top-5 / top-8 contienen la fuente correcta

test_queries = [
    ("alto riesgo reglamento IA", "eu_ai_act"),
    ("supervisión evaluación AESIA sistemas IA", "aesia"),
    ("tratamiento datos personales base legal RGPD", "lopd_rgpd"),
    ("ley organica proteccion datos 3/2018", "boe"),
]

ks = [3, 5, 8]

for query, expected_source in test_queries:
    print("\n" + "="*80)
    print("QUERY:", query)
    print("Esperado:", expected_source)

    for k in ks:
        results = chroma_search(query, k=k)
        sources = [meta.get("source") for _, _, meta in results]
        hit = expected_source in sources

        print(f"k={k} → sources={sources} → HIT={hit}")


QUERY: alto riesgo reglamento IA
Esperado: eu_ai_act
k=3 → sources=['aesia', 'boe', 'lopd_rgpd'] → HIT=False
k=5 → sources=['aesia', 'boe', 'lopd_rgpd', 'boe', 'aesia'] → HIT=False
k=8 → sources=['aesia', 'boe', 'lopd_rgpd', 'boe', 'aesia', 'aesia', 'aesia', 'lopd_rgpd'] → HIT=False

QUERY: supervisión evaluación AESIA sistemas IA
Esperado: aesia
k=3 → sources=['aesia', 'boe', 'lopd_rgpd'] → HIT=True
k=5 → sources=['aesia', 'boe', 'lopd_rgpd', 'aesia', 'aesia'] → HIT=True
k=8 → sources=['aesia', 'boe', 'lopd_rgpd', 'aesia', 'aesia', 'aesia', 'boe', 'lopd_rgpd'] → HIT=True

QUERY: tratamiento datos personales base legal RGPD
Esperado: lopd_rgpd
k=3 → sources=['aesia', 'boe', 'lopd_rgpd'] → HIT=True
k=5 → sources=['aesia', 'boe', 'lopd_rgpd', 'boe', 'lopd_rgpd'] → HIT=True
k=8 → sources=['aesia', 'boe', 'lopd_rgpd', 'boe', 'lopd_rgpd', 'aesia', 'aesia', 'aesia'] → HIT=True

QUERY: ley organica proteccion datos 3/2018
Esperado: boe
k=3 → sources=['aesia', 'boe', 'boe'] → HIT=True
k=5 → s

In [22]:
# === Conclusión automática: mejor k ===

score_k = {k: 0 for k in ks}

for query, expected_source in test_queries:
    for k in ks:
        results = chroma_search(query, k=k)
        sources = [meta.get("source") for _, _, meta in results]
        if expected_source in sources:
            score_k[k] += 1

print("Resultados por k:")
for k, score in score_k.items():
    print(f"k={k} → {score}/{len(test_queries)} aciertos")

best_k = max(score_k, key=score_k.get)
print("\nMejor k según pruebas:", best_k)

Resultados por k:
k=3 → 3/4 aciertos
k=5 → 3/4 aciertos
k=8 → 3/4 aciertos

Mejor k según pruebas: 3


In [24]:
# === Priorización suave por dominio (sin excluir fuentes) ===


def detect_main_source(query: str):
    q = query.lower()
    if any(w in q for w in ["alto riesgo", "reglamento ia", "ai act", "reglamento europeo"]):
        return "eu_ai_act"
    if "aesia" in q:
        return "aesia"
    if any(w in q for w in ["rgpd", "datos personales", "tratamiento datos"]):
        return "lopd_rgpd"
    if any(w in q for w in ["ley organica", "3/2018", "boe"]):
        return "boe"
    return None


def soft_priority_search(query: str, k: int = 3, base_k: int = 8):
    main_source = detect_main_source(query)
    print("Fuente prioritaria detectada:", main_source)

    # 1️⃣ Recuperación global
    base_results = chroma_search(query, k=base_k)

    reranked = []
    for dist, doc, meta in base_results:
        source = meta.get("source")
        bonus = 0

        # 2️⃣ Bonus suave si coincide con fuente principal
        if main_source and source == main_source:
            bonus = 0.8   # puedes ajustar entre 0.5–1.0

        # menor distance es mejor → lo invertimos suavemente
        score = (-dist) + bonus
        reranked.append((score, dist, doc, meta))

    # 3️⃣ Reordenar por score
    reranked.sort(reverse=True, key=lambda x: x[0])

    # devolver top-k final
    final = [(dist, doc, meta) for score, dist, doc, meta in reranked[:k]]
    return final


# === Prueba ===
query = "¿Qué requisitos tiene un sistema de IA de alto riesgo según el Reglamento de IA?"
results = soft_priority_search(query, k=3)
pretty_print(results, max_chars=260)

Fuente prioritaria detectada: eu_ai_act

#1  distance=2.8709  source=aesia  title=MINISTERIO PARA LA  
MINISTERIO PARA LA TRANSFORMACIÓN DIGITAL Y DE LA FUNCIÓN PÚBLICA SECRETARÍA DE ESTADO DE DIGITALIZACIÓN E INTELIGENCIA ARTIFICIAL ESPAÑA DIGITAL 2030 E INTERNACIONALIZACIÓN Este documento es de carácter confidencial y tiene efectos meramente informativos. No
--------------------------------------------------------------------------------

#2  distance=3.3300  source=boe  title=BOE-A-2018-16673 Ley Orgánica 3/2018, de 5 de diciembre, de Protección de Datos Personales y garantía de los derechos digitales.  
Las actuaciones de investigación podrán realizarse a través de sistemas digitales que, mediante la videoconferencia u otro sistema similar, permitan la comunicación bidireccional y simultánea de imagen y sonido, la interacción visual, auditiva y verbal entre l
--------------------------------------------------------------------------------

#3  distance=3.6669  source=lopd_rgpd  tit

# Conclusiones – Embeddings y Vector Store (Chroma)

## 1. Objetivo del notebook

El objetivo de este notebook fue:

- Generar embeddings de los chunks documentales.
- Construir un vector store persistente en Chroma.
- Evaluar el comportamiento del sistema de recuperación (retrieval).
- Ajustar parámetros como k y explorar estrategias de filtrado y priorización.

Este notebook representa la fase de indexación y recuperación semántica (capa de retrieval dentro de la arquitectura RAG).

---

## 2. Construcción del Vector Store

Se realizó:

- Chunking previo de los documentos legales.
- Generación de embeddings (dimensión 384).
- Persistencia en Chroma.
- Inserción en batches de 200 registros.
- Verificación del count final.

Resultado:

- embeddings.shape = (699, 384)
- col.count() = 699

Se confirmó que la indexación fue correcta, consistente y persistente.

---

## 3. Evaluación del Retrieval y ajuste de k

Se probaron múltiples valores de k (3, 5 y 8) con queries representativas:

- AI Act (alto riesgo)
- Supervisión y evaluación AESIA
- RGPD (tratamiento de datos personales)
- Ley Orgánica 3/2018

Resultados obtenidos:

k=3 → 3/4 aciertos  
k=5 → 3/4 aciertos  
k=8 → 3/4 aciertos  

Conclusiones:

- Aumentar k no mejoró el rendimiento.
- k=3 es suficiente y más eficiente.
- El sistema presenta buena precisión en top-3.
- Se selecciona k=3 como valor óptimo para este proyecto.

---

## 4. Observaciones relevantes detectadas

Durante las pruebas se observó que:

- Algunas queries sobre AI Act no priorizaban chunks del corpus eu_ai_act.
- Otros corpus (AESIA, BOE, RGPD) aparecían antes debido a mayor proximidad semántica según el embedding.

Este comportamiento no constituye un error técnico, sino una consecuencia normal del retrieval denso basado en similitud vectorial.

Indica que:

- El modelo de embeddings prioriza similitud semántica global.
- El chunking del AI Act (procedente de XML estructurado) puede introducir cierto ruido.
- La señal semántica de otros corpus puede ser más fuerte en determinados términos.

---

## 5. Estrategias evaluadas

Se probaron tres enfoques de control del retrieval:

### A) Filtro duro por dominio (where)

- Recupera exclusivamente un corpus específico.
- Garantiza precisión por fuente.
- Elimina la complementariedad entre normas.

Conclusión: útil en escenarios muy controlados, pero demasiado restrictivo para un sistema general.

---

### B) Retrieval global sin filtros

- Permite complementariedad entre corpus.
- Puede mezclar normas incluso cuando la intención de la query es específica.

Conclusión: buen comportamiento general, pero sin control explícito por dominio.

---

### C) Priorización suave por dominio (implementada)

Se implementó un reranking ligero:

- Recuperación top-8 global.
- Asignación de un bonus de score si coincide con el dominio principal detectado.
- Reordenamiento sin exclusión de otras fuentes.

Conclusión:

- Mantiene la complementariedad.
- Permite cierto control semántico.
- No fuerza resultados artificialmente.
- Es una solución elegante y metodológicamente defendible.

---

## 6. Decisiones finales adoptadas

Se decide:

- Mantener los embeddings actuales.
- No rehacer el chunking (alcance adecuado para una Prueba de Concepto).
- Utilizar k=3 como configuración por defecto.
- Incorporar priorización suave como mejora opcional.
- Documentar las limitaciones propias del retrieval denso.

---

## 7. Nivel de madurez del sistema

Para un proyecto final de máster:

- La arquitectura es correcta.
- El vector store está correctamente construido.
- La evaluación es reproducible.
- Las decisiones están justificadas.
- Se exploraron mejoras sin sobreingeniería.

No se trata de un sistema productivo industrial, pero sí de una Prueba de Concepto técnicamente sólida y defendible.

---

## 8. Posibles mejoras en un entorno productivo

En un entorno real de producción podrían incorporarse:

- Retrieval híbrido (BM25 + Dense retrieval).
- Mejor normalización del XML del AI Act.
- Métricas cuantitativas formales (Precision@k, Recall@k).
- Evaluación automatizada con conjunto de validación etiquetado.

Para el alcance del proyecto final, pensamos que la solución implementada es coherente, suficiente y metodológicamente adecuada.